# Sync Score Analysis for Deepfake Detection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold, GroupKFold
from sklearn.utils import shuffle
from pathlib import Path
import ast
import re
import warnings
warnings.filterwarnings('ignore')

CSV_PATH = '../debug-dump/eval_results_20260127_110901.csv'
OUTPUT_DIR = Path('./analysis_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Data Loading

In [ ]:
df = pd.read_csv(CSV_PATH)

def get_method(vid):
    if 'FakeVideo-FakeAudio' in str(vid): return 'FakeVideo+FakeAudio'
    elif 'FakeVideo-RealAudio' in str(vid): return 'FakeVideo-RealAudio'
    elif 'RealVideo-FakeAudio' in str(vid): return 'RealVideo-FakeAudio'
    elif 'RealVideo-RealAudio' in str(vid): return 'Real'
    return 'Unknown'

df['method'] = df['video_id'].apply(get_method)
df['label'] = df['method'].apply(lambda x: 1 if 'Fake' in x else 0)
df['label_str'] = df['label'].map({0: 'Real', 1: 'Fake'})

print(f"Total: {len(df)}, Real: {(df['label']==0).sum()}, Fake: {(df['label']==1).sum()}")
print(df['method'].value_counts())

def parse_sims(s):
    try: return np.array(ast.literal_eval(s))
    except: return None

if 'per_frame_sims' in df.columns:
    df['sims'] = df['per_frame_sims'].apply(parse_sims)
    print(f"Per-frame: {df['sims'].notna().sum()} samples, {len(df[df['sims'].notna()]['sims'].iloc[0])} frames")

SYNC_FEATURES = ['sync_mean', 'sync_min', 'sync_max', 'sync_std', 'sync_p10', 'sync_p25', 'sync_p50']
INTRA_FEATURES = ['intra_sim_audio', 'intra_sim_visual']
DISTANCE_FEATURES = ['sync_euc', 'sync_pearson']
ALL_FEATURES = [f for f in SYNC_FEATURES + INTRA_FEATURES + DISTANCE_FEATURES if f in df.columns]

## 2. Statistical Comparison

In [ ]:
real_df = df[df['label'] == 0]
fake_df = df[df['label'] == 1]

stats_data = []
for feat in ALL_FEATURES:
    real_vals = real_df[feat].dropna()
    fake_vals = fake_df[feat].dropna()
    t_stat, p_val = stats.ttest_ind(real_vals, fake_vals)
    pooled_std = np.sqrt((real_vals.std()**2 + fake_vals.std()**2) / 2)
    cohens_d = (real_vals.mean() - fake_vals.mean()) / pooled_std if pooled_std > 0 else 0
    auc = roc_auc_score(df['label'], df[feat])
    direction = 'Higher=Fake' if auc >= 0.5 else 'Lower=Fake'
    auc = auc if auc >= 0.5 else 1 - auc
    stats_data.append({'Feature': feat, 'Real': real_vals.mean(), 'Fake': fake_vals.mean(),
                       'Cohen_d': cohens_d, 'p': p_val, 'AUC': auc, 'Dir': direction})

stats_df = pd.DataFrame(stats_data).sort_values('AUC', ascending=False)
stats_df

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, metric in enumerate(stats_df.head(6)['Feature'].tolist()):
    ax = axes[i]
    auc_val = stats_df[stats_df['Feature']==metric]['AUC'].values[0]
    sns.histplot(data=df, x=metric, hue='label_str', stat='density', kde=True, ax=ax,
                 hue_order=['Real', 'Fake'], common_norm=False,
                 palette={'Real': '#2ecc71', 'Fake': '#e74c3c'}, alpha=0.6)
    ax.set_title(f'{metric} (AUC: {auc_val:.3f})')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_distributions.png', dpi=150)

## 3. Temporal Pattern Investigation

In [ ]:
df_valid = df[df['sims'].notna()].copy()

def extract_temporal_features(sims):
    if sims is None or len(sims) < 4: return None
    n = len(sims)
    first_half, second_half = sims[:n//2], sims[n//2:]
    slope = stats.linregress(np.arange(n), sims)[0]
    return {
        'trend_slope': slope,
        'first_half': np.mean(first_half),
        'second_half': np.mean(second_half),
        'half_diff': np.mean(second_half) - np.mean(first_half),
        'first_frame': sims[0],
        'std': np.std(sims),
        'mean': np.mean(sims),
    }

temporal_features = pd.DataFrame([extract_temporal_features(s) for s in df_valid['sims']])
temporal_features['label'] = df_valid['label'].values
temporal_features['label_str'] = df_valid['label_str'].values
temporal_features['method'] = df_valid['method'].values

real_tf = temporal_features[temporal_features['label'] == 0]
fake_tf = temporal_features[temporal_features['label'] == 1]

print(f"Real half_diff: {real_tf['half_diff'].mean():.4f}")
print(f"Fake half_diff: {fake_tf['half_diff'].mean():.4f}")
print(f"Corr(trend_slope, half_diff): {temporal_features['trend_slope'].corr(temporal_features['half_diff']):.4f}")

for feat in ['trend_slope', 'half_diff', 'std']:
    auc = roc_auc_score(temporal_features['label'], temporal_features[feat])
    print(f"{feat}: AUC={max(auc, 1-auc):.4f}")

In [ ]:
# Temporal feature distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, feat in enumerate(['trend_slope', 'half_diff', 'std']):
    ax = axes[i]
    auc = max(roc_auc_score(temporal_features['label'], temporal_features[feat]),
              1-roc_auc_score(temporal_features['label'], temporal_features[feat]))
    sns.histplot(data=temporal_features, x=feat, hue='label_str', stat='density', kde=True, ax=ax,
                 hue_order=['Real', 'Fake'], common_norm=False,
                 palette={'Real': '#2ecc71', 'Fake': '#e74c3c'}, alpha=0.6)
    note = ' *' if feat in ['trend_slope', 'half_diff'] else ''
    ax.set_title(f'{feat} (AUC: {auc:.3f}){note}')

plt.suptitle('* trend_slope and half_diff may be dataset-specific', fontsize=10, y=0.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'temporal_feature_distributions.png', dpi=150)

In [ ]:
corr_slope_halfdiff = temporal_features['trend_slope'].corr(temporal_features['half_diff'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=temporal_features, x='label_str', y='half_diff', ax=axes[0], 
            order=['Real', 'Fake'],
            palette={'Real': '#2ecc71', 'Fake': '#e74c3c'})
axes[0].set_xlabel('')
axes[0].set_title('Half Diff (Second - First Half)')
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)

sns.scatterplot(data=temporal_features, x='half_diff', y='trend_slope', 
                hue='label_str', hue_order=['Real', 'Fake'],
                alpha=0.5, ax=axes[1], palette={'Real': '#2ecc71', 'Fake': '#e74c3c'})
axes[1].set_title(f'Trend Slope vs Half Diff (r={corr_slope_halfdiff:.3f})')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'temporal_patterns.png', dpi=150)

### Possible causes for temporal pattern
1. Video padding (if fake videos are shorter, padding may affect sync scores)
2. Deepfake generation pipeline artifacts (audio splicing, "warm-up" period)
3. Dataset construction artifacts

TODO: Check original video durations to verify/rule out padding hypothesis

In [ ]:
# Check video durations if available
# For now, we can check the distribution of first_frame sync scores as a proxy
# (lower first_frame sync might indicate padding with silence/black frames)

print("First frame sync scores by class:")
print(f"  Real: {real_tf['first_frame'].mean():.4f} +/- {real_tf['first_frame'].std():.4f}")
print(f"  Fake: {fake_tf['first_frame'].mean():.4f} +/- {fake_tf['first_frame'].std():.4f}")

# Check by method
print("\nFirst frame sync by method:")
print(temporal_features.groupby('method')['first_frame'].agg(['mean', 'std']).round(4))

## 4. First-Frame Removal

In [ ]:
def extract_features_skip(sims, skip=0):
    sims = sims[skip:]
    if len(sims) < 4: return None
    n = len(sims)
    return {
        'trend_slope': stats.linregress(np.arange(n), sims)[0],
        'half_diff': np.mean(sims[n//2:]) - np.mean(sims[:n//2]),
        'std': np.std(sims),
        'mean': np.mean(sims),
    }

print(f"{'Skip':>6} {'trend_slope':>12} {'half_diff':>12} {'std':>12} {'mean':>12}")
for skip in [0, 1, 2, 3]:
    feats_list = [{'label': row['label'], **extract_features_skip(row['sims'], skip)} 
                  for _, row in df_valid.iterrows() if extract_features_skip(row['sims'], skip)]
    feats = pd.DataFrame(feats_list)
    aucs = {m: max(roc_auc_score(feats['label'], feats[m]), 1-roc_auc_score(feats['label'], feats[m]))
            for m in ['trend_slope', 'half_diff', 'std', 'mean']}
    print(f"{skip:>6} {aucs['trend_slope']:12.4f} {aucs['half_diff']:12.4f} {aucs['std']:12.4f} {aucs['mean']:12.4f}")

## 5. Feature Engineering

In [ ]:
def extract_advanced_features(sims):
    if sims is None or len(sims) < 4: return None
    n = len(sims)
    diffs = np.diff(sims)
    slope, _, r, _, _ = stats.linregress(np.arange(n), sims)
    centered = sims - np.mean(sims)
    var = np.var(sims)
    autocorr = np.correlate(centered[:-1], centered[1:])[0] / (var * (n-1)) if var > 1e-10 else 0
    return {
        'mean': np.mean(sims), 'std': np.std(sims), 'min': np.min(sims), 'max': np.max(sims),
        'range': np.max(sims) - np.min(sims), 'skewness': stats.skew(sims), 'kurtosis': stats.kurtosis(sims),
        'p25': np.percentile(sims, 25), 'p75': np.percentile(sims, 75), 'p95': np.percentile(sims, 95),
        'iqr': np.percentile(sims, 75) - np.percentile(sims, 25),
        'jitter_mean': np.mean(np.abs(diffs)), 'jitter_std': np.std(diffs), 'jitter_max': np.max(np.abs(diffs)),
        'trend_slope': slope, 'trend_r2': r**2,
        'half_diff': np.mean(sims[n//2:]) - np.mean(sims[:n//2]),
        'autocorr_lag1': autocorr,
    }

advanced_features = pd.DataFrame([extract_advanced_features(s) for s in df_valid['sims']])
advanced_features['label'] = df_valid['label'].values
advanced_features['label_str'] = df_valid['label_str'].values

DERIVED = ['mean', 'std', 'min', 'max', 'range', 'skewness', 'kurtosis', 'p25', 'p75', 'p95', 'iqr',
           'jitter_mean', 'jitter_std', 'jitter_max', 'trend_slope', 'trend_r2', 'half_diff', 'autocorr_lag1']

# Features that may be dataset-specific (require cross-dataset validation)
TEMPORAL_FEATURES = ['trend_slope', 'half_diff']

print(f"{'Feature':<20} {'AUC':>8} {'Direction':>15} {'Note':>12}")
for feat in DERIVED:
    if feat in advanced_features.columns:
        auc = roc_auc_score(advanced_features['label'], advanced_features[feat])
        direction = 'Higher=Fake' if auc >= 0.5 else 'Lower=Fake'
        auc_adj = max(auc, 1-auc)
        note = 'temporal' if feat in TEMPORAL_FEATURES else ''
        print(f"{feat:<20} {auc_adj:8.4f} {direction:>15} {note:>12}")

In [ ]:
# All derived feature distributions (top 9 by AUC)
feature_aucs = [(f, max(roc_auc_score(advanced_features['label'], advanced_features[f]),
                        1-roc_auc_score(advanced_features['label'], advanced_features[f])))
                for f in DERIVED if f in advanced_features.columns]
feature_aucs.sort(key=lambda x: -x[1])

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
for i, (feat, auc) in enumerate(feature_aucs[:9]):
    ax = axes[i]
    sns.histplot(data=advanced_features, x=feat, hue='label_str', stat='density', kde=True, ax=ax,
                 hue_order=['Real', 'Fake'], common_norm=False,
                 palette={'Real': '#2ecc71', 'Fake': '#e74c3c'}, alpha=0.6)
    note = ' *' if feat in TEMPORAL_FEATURES else ''
    ax.set_title(f'{feat} (AUC: {auc:.3f}){note}')
plt.suptitle('* temporal features may be dataset-specific', fontsize=10, y=0.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'all_feature_distributions.png', dpi=150)

## 6. Robustness Validation

In [ ]:
# Balanced subset
n_real = (advanced_features['label'] == 0).sum()
balanced_fake = advanced_features[advanced_features['label'] == 1].sample(n=n_real, random_state=42)
balanced_df = pd.concat([advanced_features[advanced_features['label'] == 0], balanced_fake])

print("Balanced Subset:")
for feat in ['trend_slope', 'half_diff', 'std']:
    full = max(roc_auc_score(advanced_features['label'], advanced_features[feat]),
               1-roc_auc_score(advanced_features['label'], advanced_features[feat]))
    bal = max(roc_auc_score(balanced_df['label'], balanced_df[feat]),
              1-roc_auc_score(balanced_df['label'], balanced_df[feat]))
    print(f"   {feat}: Full={full:.4f}, Balanced={bal:.4f}")

In [ ]:
# Subject-level CV
def extract_subject(vid):
    match = re.search(r'id\d+', str(vid))
    return match.group() if match else str(vid)

df_valid['subject'] = df_valid['video_id'].apply(extract_subject)
advanced_features['subject'] = df_valid['subject'].values

X = advanced_features[['std', 'mean']].values
y = advanced_features['label'].values
groups = advanced_features['subject'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
lr = LogisticRegression(class_weight='balanced', max_iter=1000)

standard_cv = cross_val_score(lr, X_scaled, y, cv=5, scoring='roc_auc')
print(f"Standard 5-Fold: {standard_cv.mean():.4f} +/- {standard_cv.std():.4f}")

try:
    group_cv = GroupKFold(n_splits=5)
    group_scores = cross_val_score(lr, X_scaled, y, cv=group_cv, groups=groups, scoring='roc_auc')
    print(f"Subject-Grouped: {group_scores.mean():.4f} +/- {group_scores.std():.4f}")
except Exception as e:
    print(f"Subject-Grouped CV error: {e}")

In [ ]:
# Permutation test
n_perms = 100
true_auc = max(roc_auc_score(advanced_features['label'], advanced_features['std']),
               1-roc_auc_score(advanced_features['label'], advanced_features['std']))
perm_aucs = [max(roc_auc_score(shuffle(advanced_features['label'], random_state=i), advanced_features['std']),
                 1-roc_auc_score(shuffle(advanced_features['label'], random_state=i), advanced_features['std']))
             for i in range(n_perms)]
p_value = np.mean(np.array(perm_aucs) >= true_auc)
print(f"True AUC: {true_auc:.4f}, Perm AUC: {np.mean(perm_aucs):.4f}, p={p_value:.4f}")

## 7. Model Comparison

In [ ]:
# Conservative features (excluding temporal trend features)
CONSERVATIVE = ['std', 'mean', 'jitter_mean', 'jitter_std', 'kurtosis', 'iqr']
CONSERVATIVE = [f for f in CONSERVATIVE if f in advanced_features.columns]

X = advanced_features[CONSERVATIVE].fillna(0).values
y = advanced_features['label'].values
X_scaled = StandardScaler().fit_transform(X)

models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=1000),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Conservative features: {CONSERVATIVE}\n")

best_name, best_auc = None, 0
for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='roc_auc')
    print(f"{name}: {scores.mean():.4f} +/- {scores.std():.4f}")
    if scores.mean() > best_auc:
        best_auc, best_name = scores.mean(), name
print(f"\nBest: {best_name} ({best_auc:.4f})")

In [ ]:
# Compare: ALL features vs conservative features
ALL_DERIVED = [f for f in DERIVED if f in advanced_features.columns]
X_all = advanced_features[ALL_DERIVED].fillna(0).values
X_all_scaled = StandardScaler().fit_transform(X_all)

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
scores_all = cross_val_score(rf, X_all_scaled, y, cv=cv, scoring='roc_auc')
scores_conservative = cross_val_score(rf, X_scaled, y, cv=cv, scoring='roc_auc')

print(f"RandomForest with ALL features (CV):          {scores_all.mean():.4f}")
print(f"RandomForest with conservative features (CV): {scores_conservative.mean():.4f}")

## 8. ROC Curves (using CV predictions)

In [ ]:
# Use cross_val_predict to get out-of-fold predictions for honest ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Conservative features
ax = axes[0]
for feat in ['std', 'jitter_mean', 'mean']:
    if feat in advanced_features.columns:
        auc = roc_auc_score(y, advanced_features[feat])
        score = advanced_features[feat] if auc >= 0.5 else -advanced_features[feat]
        auc = max(auc, 1-auc)
        fpr, tpr, _ = roc_curve(y, score)
        ax.plot(fpr, tpr, label=f'{feat} ({auc:.3f})', linewidth=2, alpha=0.7)

# Get out-of-fold predictions for combined model
best_model = models[best_name]
y_prob_cv = cross_val_predict(best_model, X_scaled, y, cv=cv, method='predict_proba')[:, 1]
combined_auc = roc_auc_score(y, y_prob_cv)
fpr, tpr, _ = roc_curve(y, y_prob_cv)
ax.plot(fpr, tpr, label=f'Combined ({combined_auc:.3f})', linewidth=3, linestyle='--', color='black')
ax.plot([0, 1], [0, 1], 'k:', alpha=0.5)
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('Conservative Features')
ax.legend(loc='lower right')

# Right: Including temporal features
ax = axes[1]
for feat in ['trend_slope', 'half_diff', 'std']:
    if feat in advanced_features.columns:
        auc = roc_auc_score(y, advanced_features[feat])
        score = advanced_features[feat] if auc >= 0.5 else -advanced_features[feat]
        auc = max(auc, 1-auc)
        fpr, tpr, _ = roc_curve(y, score)
        linestyle = '--' if feat in TEMPORAL_FEATURES else '-'
        ax.plot(fpr, tpr, label=f'{feat} ({auc:.3f})', linewidth=2, alpha=0.7, linestyle=linestyle)

# Get out-of-fold predictions for all features model
rf_all = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
y_prob_all_cv = cross_val_predict(rf_all, X_all_scaled, y, cv=cv, method='predict_proba')[:, 1]
all_auc = roc_auc_score(y, y_prob_all_cv)
fpr, tpr, _ = roc_curve(y, y_prob_all_cv)
ax.plot(fpr, tpr, label=f'All features ({all_auc:.3f})', linewidth=3, linestyle='--', color='black')
ax.plot([0, 1], [0, 1], 'k:', alpha=0.5)
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('Including Temporal Features')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curves.png', dpi=150)